# Разбор A. Исследовательский вопрос

В предыдущем уроке мы отделили объект от его описания. Теперь лаборатория спрашивает, можно ли выбирать растворитель по прогнозу растворимости. У истрадефиллина нет в этом наборе сопоставимого измерения: обещать точность для него нельзя. Вспомогательная ESOL позволяет сформулировать опровержимую гипотезу о методе. До вычисления предположите, будет ли ошибка одинаковой у хорошо и плохо растворимых соединений. Для B вернитесь к аторвастатину: определите форму, pH, температуру, равновесную или кинетическую растворимость, а не переносите числа ESOL на лекарство.

Прочитайте закреплённый CSV и сохраните row_id. Не сортируйте прогнозы отдельно от наблюдений. Рассчитайте остатки, сравните средние ошибки для y < −3 и y ≥ −3. Постройте наблюдение–прогноз и диагональ. Затем выпишите до следующего опыта endpoint, критерий ошибки и правила исключения.

In [2]:
from pathlib import Path
import sys,json,math
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'programm/course.json').is_file())
sys.path.insert(0,str(ROOT/'practicuum/shared'))
from course_support import DATA,compound,esol,workspace,write_json,init_dossier
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
def publish_plot(fig,name):
    fig.savefig(OUT/name,dpi=120,bbox_inches='tight')
    display(fig)
    plt.close(fig)
OUT=workspace('M01-L02','case')


Перед вами контрольный набор ESOL: пары наблюдаемых и предсказанных logS. Сначала спросим, насколько ошибается прогноз в среднем. До запуска выберите, какая метрика сильнее отреагирует на редкие крупные ошибки — MAE или RMSE — и объясните свой прогноз. Эти данные относятся к растворимости; в них нет измерения проницаемости ГЭБ.

Я сначала превращаю общее желание «предсказывать растворимость» в проверяемое правило: на этих опубликованных наблюдениях вычислить ошибку сохранённых прогнозов. Каждая ошибка — разность двух значений одной строки. Если потерять соответствие строк, формулы MAE и RMSE всё равно дадут числа, поэтому смысл входных пар проверяется раньше метрики.

Почему считаю обе величины? Две ошибки +1 и −1 в условном примере имеют среднее ноль, хотя оба прогноза ошибочны. Модуль и квадрат устраняют это взаимное погашение; квадрат дополнительно усиливает крупные отклонения. Я выбираю показатель по тому, какие ошибки важны для решения, и смотрю, насколько вывод зависит от выбора показателя.

In [3]:
rows=esol()
errors=[r['predicted']-r['measured'] for r in rows]
mae=sum(abs(e) for e in errors)/len(errors)
rmse=math.sqrt(sum(e*e for e in errors)/len(errors))
print({'n':len(rows),'MAE_logS':mae,'RMSE_logS':rmse})


{'n': 1128, 'MAE_logS': 0.6978617021276595, 'RMSE_logS': 0.9101322725549837}


Одного среднего недостаточно для решения, где модель пригодна. Ниже разделяем записи по заранее указанному порогу наблюдаемого logS и смотрим, одинаковы ли ошибки. На диаграмме точки около диагонали означают близость прогноза и наблюдения; удаление от неё требует объяснения. Сравнение групп описательное и не заменяет независимого тестирования.

In [4]:
for label,group in [('low',[r for r in rows if r['measured'] < -3]),('high',[r for r in rows if r['measured'] >= -3])]:
    print(label,len(group),sum(abs(r['predicted']-r['measured']) for r in group)/len(group))
fig,ax=plt.subplots()
ax.scatter([r['measured'] for r in rows],[r['predicted'] for r in rows],s=7,alpha=.4)
ax.plot([-12,3],[-12,3],color='black');ax.set(xlabel='Observed logS',ylabel='Predicted logS')
publish_plot(fig,'esol.png')


low 543 0.744340699815838
high 585 0.6547196581196582


<Figure size 640x480 with 1 Axes>

В досье сохраняем вычисленные метрики контрольного набора. Для истрадефиллина и аторвастатина остаётся отдельный вопрос: совпадают ли endpoint, условия и область применимости с задачей, ради которой выбирается кандидат?

In [5]:
result={'n':len(rows),'mae_logS':mae,'rmse_logS':rmse,'kind':'calculated','target_drug_prediction':'NOT_RUN'}


Дальше я проверяю, не скрывает ли среднее различие областей logS. Разделение по наблюдаемому значению помогает описать этот контрольный набор, но при прогнозе нового вещества истинный logS заранее неизвестен. Поэтому результат сравнения групп не превращается автоматически в рабочее правило выбора модели для неизвестного кандидата. Следующее действие — сформулировать проверяемое ограничение и подобрать независимые данные нужного endpoint.

In [6]:
write_json(OUT/'result.json',result)
from dossier_records import append_record
receipt=append_record(init_dossier(),'M01-L02','A',result,ROOT,inputs=['practicuum/shared/data/esol.csv'])
print('Versioned evidence:',receipt)


Versioned evidence: C:\Users\Miakhlov\Documents\GitHub\Computational chemistry_course\tmp\reasoning-full-20261120\workspace\my_research\evidence\M01-L02\20261120T105306043696Z-A-eb2cb67ed96b4beb899e1c26c91392cd.json


Расположение относительно диагонали показывает направление ошибки, но не причину. Корреляция не гарантирует калибровку. Для решения об истрадефиллине нужен новый релевантный набор измерений. Вопрос для перехода: каким образом другой исследователь воспроизведёт выбор строк и версии расчёта?

Если общая RMSE выглядит приемлемой, а в одной группе ошибка больше, как изменится ваше исследовательское решение? Назовите недостающее свидетельство, прежде чем переносить результат на B.

# M01. Как наставник читает первичную работу

Пока следующая вычислительная задача ещё не выбрана, вернёмся к источнику: **Crevar-Sakač et al. (2013). An Improved HPLC Method with the Aid of a Chemometric Protocol: Simultaneous Determination of Atorvastatin and Its Metabolites in Plasma**. [Локальный полный текст](../../../../../shared/data/source-responses/hplc_article.xml), [DOI](https://doi.org/10.3390/molecules18032469). Откройте **Abstract, Introduction и назначение метода**.

Выберите предложение о назначении аналитического метода; отделите объект анализа, матрицу и измеряемый результат.

Наставник начинает с вопроса «что здесь непосредственно наблюдали?». Найдите одно такое наблюдение и его место в источнике. Затем рядом запишите трактовку авторов и ещё одно возможное объяснение. Это учебный приём чтения, а не реконструкция мыслей авторов. Если численного ряда нет, вывод нельзя усиливать придуманными точками.

Перед вычислением назовите объект, условия, величину и единицы. После вычисления сравните ответ с исходным утверждением. При расхождении сначала проверьте форму вещества, основу концентрации и преобразования таблицы; только затем обсуждайте модель. Смысл операции — получить проверяемый аргумент для следующего решения досье.

Теперь перенесите приём на [самостоятельную мини-рецензию B](../steps/03_article_review.md). Для проверки собственного вывода используйте вопрос: **Можно ли из описания метода анализа плазмы заключить, что вещество избирательно накапливается в печени?**